# 03 · Análise de sentimentos com FinBERT-PT-BR

Entrega da Etapa 2 (22/10). O notebook:

1. carrega as manchetes revisadas (`teste_revisao.xlsx`, montada a partir da coleta do notebook 01);
2. classifica cada uma como **boa, ruim ou neutra para o acionista** com o modelo pré-treinado FinBERT-PT-BR (`lucas-leme/FinBERT-PT-BR`), sem ajuste;
3. testa exemplos de cada classe, inclusive manchetes escritas por nós;
4. registra os erros e termina com uma breve análise.

Os rótulos das manchetes foram sugeridos por IA generativa (Claude) e revisados por nós seguindo o guia de rotulagem (`Entregas/2026-10-22/guia_rotulagem.md`).

**Como rodar no Colab:** GPU T4, depois *Executar tudo*. Antes, suba a `teste_revisao.xlsx` para `MyDrive/portfolio-radar/`.

In [ ]:
try:
    import google.colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

from pathlib import Path

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA = Path("/content/drive/MyDrive/portfolio-radar")
else:
    PASTA = Path("raw")

## Manchetes

In [ ]:
import pandas as pd

ROTULOS = ["boa", "neutra", "ruim"]

manchetes = pd.read_excel(PASTA / "teste_revisao.xlsx")
manchetes["rotulo"] = manchetes["rotulo"].str.strip().str.lower()

invalidos = manchetes[~manchetes["rotulo"].isin(ROTULOS + ["descartar"])]
assert invalidos.empty, f"Rótulos inválidos nas linhas: {invalidos['id'].tolist()}"

manchetes = manchetes[manchetes["rotulo"] != "descartar"].reset_index(drop=True)
print(f"{len(manchetes)} manchetes")
manchetes["rotulo"].value_counts().to_frame("manchetes")

## Classificação com FinBERT-PT-BR

In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
clf = pipeline("text-classification", model="lucas-leme/FinBERT-PT-BR", device=device)
MAPA = {"POSITIVE": "boa", "NEGATIVE": "ruim", "NEUTRAL": "neutra"}


def classifica(textos):
    saidas = clf(list(textos), batch_size=32, truncation=True)
    return [MAPA[s["label"]] for s in saidas], [round(s["score"], 2) for s in saidas]


manchetes["previsto"], manchetes["confianca"] = classifica(manchetes["manchete"])
manchetes["acertou"] = manchetes["rotulo"] == manchetes["previsto"]

pd.set_option("display.max_colwidth", 110)
print(f"Acertou {manchetes['acertou'].sum()} de {len(manchetes)}")
manchetes[["empresa", "manchete", "rotulo", "previsto", "confianca"]].head(10)

## Exemplos por classe

Para cada rótulo, alguns acertos e alguns erros.

In [ ]:
for r in ROTULOS:
    da_classe = manchetes[manchetes["rotulo"] == r]
    ok, erro = da_classe[da_classe["acertou"]], da_classe[~da_classe["acertou"]]
    print(f"=== {r} ({len(ok)} de {len(da_classe)} certas) ===")
    for _, l in pd.concat([ok.sample(min(3, len(ok)), random_state=0),
                           erro.sample(min(2, len(erro)), random_state=0)]).iterrows():
        print(f"{'OK  ' if l['acertou'] else 'ERRO'} previsto={l['previsto']:<6} {l['manchete']}")
    print()

### Manchetes escritas por nós

Casos controlados, incluindo o exemplo da proposta, em que o tom da frase e o efeito para o acionista apontam para lados diferentes.

In [ ]:
exemplos = pd.DataFrame([
    ("Itaú lucra R$ 12 bilhões no trimestre e supera expectativas", "boa"),
    ("Petrobras anuncia pagamento de R$ 20 bilhões em dividendos", "boa"),
    ("Petrobras reajusta o preço da gasolina nas refinarias", "boa"),        # ruim para o consumidor, boa para o acionista
    ("Bradesco é condenado a pagar multa milionária ao Procon", "ruim"),
    ("Ações da Ambev despencam após resultado abaixo do esperado", "ruim"),
    ("Localiza corta 2 mil vagas e fecha lojas para reduzir custos", "boa"),  # tom negativo, mas corte de custos tende a agradar o acionista
    ("WEG abre inscrições para programa de estágio", "neutra"),
    ("Vale realiza assembleia de acionistas nesta sexta-feira", "neutra"),
], columns=["manchete", "esperado"])

exemplos["previsto"], exemplos["confianca"] = classifica(exemplos["manchete"])
exemplos["acertou"] = exemplos["esperado"] == exemplos["previsto"]
exemplos

## Erros

In [ ]:
erros = manchetes[~manchetes["acertou"]]
print(f"{len(erros)} erros de {len(manchetes)}\n")
print("Rótulo esperado x previsto nos erros:")
print(erros.groupby(["rotulo", "previsto"]).size().to_string())
erros[["empresa", "manchete", "rotulo", "previsto", "confianca"]].sort_values("confianca", ascending=False)

In [ ]:
arq = PASTA / "resultado_sentimentos.xlsx"
with pd.ExcelWriter(arq) as xl:
    manchetes[["id", "empresa", "manchete", "rotulo", "previsto", "confianca", "acertou"]].to_excel(xl, sheet_name="previsoes", index=False)
    exemplos.to_excel(xl, sheet_name="exemplos_nossos", index=False)
print(f"Previsões salvas em {arq}")

## Breve análise

> **Preencher depois de rodar com as manchetes revisadas.** Os pontos em itálico foram observados numa execução preliminar e precisam ser confirmados.

**Resultado.** O FinBERT-PT-BR acertou [X] de [N] manchetes.

**Onde erra.**
- *Classifica como `ruim` notícias que consideramos neutras, como obras, manutenção e decisões judiciais ainda em andamento.* [citar 1–2 exemplos]
- *Deixa de reconhecer parte das notícias `boa`, como pagamento de dividendos, e as trata como neutras.* [confirmar]

**Por quê.** O modelo avalia o tom do texto, e o nosso critério é o impacto provável para o acionista. Os dois divergem quando a frase tem tom negativo e efeito positivo para a empresa, como reajuste de preço ou corte de custos. [citar o que aconteceu com as manchetes escritas por nós]

**Limitações.** O modelo só vê a manchete, sem o texto da matéria. Os rótulos de referência foram sugeridos por IA e revisados por nós, o que pode puxar a revisão para concordar com a sugestão.